In [5]:
import pandas as pd
import celloracle as co

In [6]:
CELL_TYPE = "Epi_Kit+Elf5+"
TF_OF_INTEREST = "Tfap2b"

safe_name = CELL_TYPE.replace("/", "_").replace("\\", "_").replace(" ", "_")
base_results = "celloracle_results/per_celltype"

# Path to your raw links (use filtered path if you prefer)
links_path = f"{base_results}/{safe_name}.celloracle.links"

In [8]:
links = co.load_hdf5(links_path)

print(f"Clusters: {links.cluster}")
print(f"\nColumns: {links.links_dict[links.cluster[0]].columns.tolist()}")
links.links_dict[links.cluster[0]].head()

Clusters: ['KO_DM', 'WT_DM']

Columns: ['source', 'target', 'coef_mean', 'coef_abs', 'p', '-logp']


,source,target,coef_mean,coef_abs,p,-logp
0,Snai2,0610040J01Rik,-0.026194,0.026194,9.541214e-08,7.020396
1,Onecut2,0610040J01Rik,0.004341,0.004341,1.738892e-02,1.759727
2,Usf1,0610040J01Rik,-0.016733,0.016733,6.879794e-08,7.162425
3,Jund,0610040J01Rik,-0.004626,0.004626,5.918482e-03,2.227790
4,Cebpd,0610040J01Rik,0.007267,0.007267,4.302011e-02,1.366328


In [9]:
all_tfs = set()
for cluster_name in links.cluster:
    all_tfs.update(links.links_dict[cluster_name]["source"].unique())

if TF_OF_INTEREST in all_tfs:
    print(f"✓ '{TF_OF_INTEREST}' found as a TF source")
else:
    print(f"⚠ '{TF_OF_INTEREST}' NOT found. Possible matches:")
    print([tf for tf in sorted(all_tfs) if TF_OF_INTEREST.lower() in tf.lower()])

✓ 'Tfap2b' found as a TF source


In [10]:
wt_cluster = [c for c in links.cluster if "WT" in c][0]
df_wt_raw = links.links_dict[wt_cluster]

df_wt = (
    df_wt_raw[df_wt_raw["source"] == TF_OF_INTEREST]
    [["source", "target", "coef_mean", "coef_abs", "p"]]
    .rename(columns={"source": "TF_source", "target": "target_gene",
                     "coef_mean": "beta_coef", "coef_abs": "beta_abs", "p": "p_value"})
    .sort_values("beta_abs", ascending=False)
    .reset_index(drop=True)
)

print(f"WT ({wt_cluster}): {len(df_wt)} targets for {TF_OF_INTEREST}")
df_wt.head(20)

WT (WT_DM): 1665 targets for Tfap2b


,TF_source,target_gene,beta_coef,beta_abs,p_value
0,Tfap2b,Slc5a7,0.362612,0.362612,1.025512e-15
1,Tfap2b,Gpc6,0.308217,0.308217,3.830442e-12
2,Tfap2b,Igfbp5,0.233750,0.233750,9.198150e-13
3,Tfap2b,Gna14,0.193390,0.193390,2.612042e-16
4,Tfap2b,Maml2,0.189887,0.189887,5.702106e-16
5,Tfap2b,Deptor,0.182518,0.182518,1.806034e-14
6,Tfap2b,Erbb4,0.181537,0.181537,1.909358e-15
7,Tfap2b,Slc28a3,0.168286,0.168286,2.816584e-15
8,Tfap2b,Egfr,0.168056,0.168056,4.067998e-18
9,Tfap2b,Rftn1,0.164314,0.164314,2.744116e-14


In [11]:
ko_cluster = [c for c in links.cluster if "KO" in c][0]
df_ko_raw = links.links_dict[ko_cluster]

df_ko = (
    df_ko_raw[df_ko_raw["source"] == TF_OF_INTEREST]
    [["source", "target", "coef_mean", "coef_abs", "p"]]
    .rename(columns={"source": "TF_source", "target": "target_gene",
                     "coef_mean": "beta_coef", "coef_abs": "beta_abs", "p": "p_value"})
    .sort_values("beta_abs", ascending=False)
    .reset_index(drop=True)
)

print(f"KO ({ko_cluster}): {len(df_ko)} targets for {TF_OF_INTEREST}")
df_ko.head(20)

KO (KO_DM): 1665 targets for Tfap2b


,TF_source,target_gene,beta_coef,beta_abs,p_value
0,Tfap2b,Igfbp5,0.612643,0.612643,1.986446e-14
1,Tfap2b,Gpc6,0.606107,0.606107,1.329005e-15
2,Tfap2b,Prkg1,-0.427560,0.427560,1.067925e-15
3,Tfap2b,Deptor,0.409937,0.409937,4.142946e-18
4,Tfap2b,Slc5a7,0.405066,0.405066,9.376216e-15
5,Tfap2b,Pde7b,0.340694,0.340694,2.212388e-17
6,Tfap2b,Rspo1,-0.309986,0.309986,1.165519e-18
7,Tfap2b,Mtmr9,0.309587,0.309587,3.806169e-16
8,Tfap2b,Thsd4,0.309584,0.309584,2.641257e-14
9,Tfap2b,Erbb4,0.303254,0.303254,3.915651e-12
